# indah 0.1.0rc1 - Colab / RunPod smoke test (R2)

This notebook is the real-hardware check for the one assumption the whole
architecture rests on: **that Server-Sent Events survive Colab's and RunPod's
HTTP proxies** (ADR-0002, requirement R2). Local tests can't prove this - only a
run on the actual proxied runtime can.

**How to run it**

- **Colab:** open this notebook in Colab and run every cell top to bottom.
- **RunPod:** in a GPU pod's JupyterLab, upload this notebook (or paste the cells
  into one) and run top to bottom. `indah.launch()` auto-detects RunPod and prints
  the `https://<pod-id>-<port>.proxy.runpod.net` URL.

Then work through the **checklist** cell near the bottom and report what passed.

## 1. Install the release candidate

An rc is a pre-release, so `pip` needs `--pre` to see it. This pulls `indah` and
its (Python-only) deps from PyPI - no Node, no build step.

In [ ]:
%pip install --pre "indah==0.1.0rc1"

## 2. Prove there's no Node in the runtime (R4)

If any of these resolve to a path, something dragged a JS toolchain into the
runtime. On a clean Colab/RunPod image they should all be absent, and `indah`
imports and serves its pre-built shell without them.

In [ ]:
import shutil
from importlib.resources import files

import indah

js_tools = ["node", "npm", "npx", "bun", "yarn", "pnpm"]
found = {t: shutil.which(t) for t in js_tools if shutil.which(t)}
print("indah version:", indah.__version__)
print("JS tools on PATH:", found or "none")

# The shell that will be served is the pre-built one shipped in the wheel.
shell = files("indah.static").joinpath("index.html").read_text(encoding="utf-8")
assert "EventSource" in shell and "api/stream" in shell
print("pre-built shell bytes:", len(shell))
print(
    "OK - indah runs with no Node needed"
    + ("  (note: JS tools are present on PATH but indah never calls them)" if found else "")
)

## 3. Launch the smoke-test app

One app that exercises every proxy-sensitive path in a single view:

1. **Streaming** - a prompt + **Generate** button streams a mock LLM reply token
   by token into a `StreamText` over SSE (the append-patch path).
2. **Live patch (slider)** - dragging the slider patches a computed label, and
   nothing else, live.
3. **Live patch (select -> dataframe)** - the select swaps the dataframe below it
   reactively.
4. **Custom component** - a registered `colorpicker` (ADR-0012) round-trips its
   value back into Python (watch the accent label update).

`launch()` embeds the app inline as an iframe and prints the proxy URL - open
that URL in a new tab too if the inline frame is blocked.

In [ ]:
import indah
from indah import (
    Button,
    Column,
    DataFrame,
    Select,
    Session,
    Signal,
    Slider,
    StreamText,
    Text,
    TextInput,
    computed,
    create_app,
)

# A custom component rendered from a declarative spec - no shell rebuild (ADR-0012).
indah.register_component(
    "colorpicker",
    render={
        "tag": "input",
        "attrs": {"type": "color"},
        "bind": {"value": "value"},
        "on": {"input": {"event": "input", "prop": "value"}},
    },
)

# 1) Streaming
prompt = Signal("Tell me about SSE")
response = StreamText(label="Response")


async def on_generate():
    response.reset()
    async for token in indah.mock_llm(prompt.value):
        response.feed(token)


# 2) Slider -> computed label
n = Signal(3)
doubled = computed(lambda: f"2 x {n.value} = {2 * n.value}   (stays live while streaming)")

# 3) Select -> DataFrame
DATASETS = {
    "Squares": {"columns": ["n", "n^2"], "rows": [[k, k * k] for k in range(1, 6)]},
    "Primes": {"columns": ["i", "prime"], "rows": [[1, 2], [2, 3], [3, 5], [4, 7]]},
}
dataset = Signal("Squares")

# 4) Custom colorpicker
accent = Signal("#5b5bd6")

page = Column(
    children=[
        Text("indah 0.1.0rc1 - R2 smoke test"),
        Text("1) Streaming - type a prompt, click Generate:"),
        TextInput(prompt, label="Prompt"),
        Button("Generate", on_click=on_generate),
        response,
        Text("2) Live patch - drag the slider (try it mid-stream):"),
        Slider(n, min=0, max=10, label="n"),
        Text(doubled),
        Text("3) Live patch - switch the dataset:"),
        Select(dataset, options=list(DATASETS), label="Dataset"),
        DataFrame(lambda: DATASETS[dataset.value], label="Data"),
        Text("4) Custom component - pick a colour:"),
        indah.custom("colorpicker", value=accent),
        Text(lambda: f"Accent colour: {accent.value}"),
    ]
)

handle = indah.launch(create_app(session=Session(page)))

## 4. Checklist - what to verify (this is the actual test)

Tick each one. If any fails, note the environment (Colab / RunPod), the browser,
and anything in the JS console or the cell output.

- [ ] **Inline iframe renders** - the app appears embedded in the cell above (or
      at the printed proxy URL) with all four sections visible.
- [ ] **Slider patches live** - dragging the slider updates the `2 x n` label
      within ~1s, and nothing else flickers.
- [ ] **Select patches live** - switching Dataset swaps the table's rows.
- [ ] **Streaming arrives incrementally** - clicking Generate fills the Response
      area word by word (many small updates), not one final dump. Dragging the
      slider *during* streaming still works - the UI never freezes.
- [ ] **Custom component round-trips** - picking a colour updates the "Accent
      colour" label to the new hex value.
- [ ] **Survives an idle gap** - leave it ~2 min, then drag the slider again; it
      still patches (the SSE heartbeat / resume held the connection through the
      proxy timeout).
- [ ] **No Node in the runtime** - section 2 reported no JS tools were needed.

**Report back:** which boxes passed on Colab, which on RunPod, and any proxy
quirks (iframe blocked, stream stalling, reconnect needed). Those go into
`docs/SLICES.md` and the R2 risk note in `docs/PLAN.md`.

## Troubleshooting

- **Inline frame is blank / blocked** - open the printed proxy URL in a new tab.
  Colab sometimes sandboxes inline iframes; the standalone tab is the fallback.
- **Stream stalls after ~100s** - that's the RunPod proxy idle cut. indah's 15s
  heartbeat plus `Last-Event-Id` resume should recover it; if a stream never
  resumes, capture the network tab and note it against ADR-0011.
- **`pip` can't find 0.1.0rc1** - it isn't published yet, or you dropped `--pre`.
  RCs are hidden from a plain `pip install indah`.